<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/fabiobento/rl-course-hf/blob/main/unit1/unit1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/fabiobento/rl-course-hf/blob/main/unit1/unit1.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

Adaptado do [repositório](https://github.com/huggingface/deep-rl-class) do [Hugging Face Deep Reinforcement Learning Course](https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt)

# Unidade 1: Treine seu primeiro agente de aprendizado por reforço profundo 🤖

Neste notebook, você treinará seu **primeiro agente de aprendizado por reforço profundo**(_Deep Reinforcement Learning agent_-Deep RL): um agente Lunar Lander que aprenderá a **pousar corretamente na Lua 🌕**.

Você usará  [Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/), uma biblioteca de Deep RL.

In [ ]:
%%html
<video controls autoplay><source src="https://huggingface.co/sb3/ppo-LunarLander-v2/resolve/main/replay.mp4" type="video/mp4"></video>

### O ambiente 🎮

- [LunarLander-v2](https://gymnasium.farama.org/environments/box2d/lunar_lander/)

### A biblioteca utilizada 📚

- [Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/)

## Uma breve recapitulação sobre Deep RL 📚

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit1/images/RL_process_game.jpg" alt="The RL process" width="100%">
<span style="font-size:80%">
Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Vamos fazer uma pequena recapitulação do que aprendemos na primeira unidade:

- O aprendizado por reforço é uma **abordagem computacional para aprender com ações**. Criamos um agente que aprende com o ambiente **interagindo com ele por meio de tentativa e erro** e recebendo recompensas (negativas ou positivas) como feedback.

- O objetivo de qualquer agente RL é **maximizar sua recompensa cumulativa esperada** (também chamada de retorno esperado), porque o RL se baseia na _hipótese da recompensa_, que é que todos os objetivos podem ser descritos como a maximização de uma recompensa cumulativa esperada.

- O processo RL é um **loop que gera uma sequência de estado, ação, recompensa e próximo estado**.

- Para calcular a recompensa cumulativa esperada (retorno esperado), **descontamos as recompensas**: as recompensas que vêm mais cedo (no início do jogo) são mais prováveis de acontecer, pois são mais previsíveis do que a recompensa futura a longo prazo.

- Para resolver um problema de RL, você deseja **encontrar uma _optimal policy_**; a política é o “cérebro” da sua IA que nos dirá qual ação tomar em um determinado estado. A ideal é aquela que fornece as ações que maximizam o retorno esperado.

- Existem **duas** maneiras de encontrar sua política ideal(_optimal policy_):

> 1. **Treinando sua política diretamente**: métodos baseados em políticas(_policy-based methods_).
> 2. **Treinando uma função de valor** que nos diz o retorno esperado que o agente obterá em cada estado e usando essa função para definir nossa política: métodos baseados em valor(_value based methods_).

- Por fim, falamos sobre RL profundo porque **introduzimos redes neurais profundas para estimar a ação a ser tomada (_policy-based_) ou para estimar o valor de um estado (_value based_), daí o nome “profundo”**.

## Instalar dependências 🔽

O primeiro passo é instalar as dependências:

- `swig`: O `swig` é necessário porque alguns ambientes do Gymnasium (como o LunarLander-v3, que usa Box2D) dependem de bibliotecas escritas em C/C++. O `swig` é uma ferramenta que gera automaticamente as interfaces entre essas bibliotecas em C/C++ e o Python, permitindo que o código Python utilize funcionalidades dessas bibliotecas nativas.
- `gymnasium[box2d]`: Contém o ambiente LunarLander-v3 🌛
- `stable-baselines3[extra]`: A biblioteca de _deep reinforcement learning_.

Pra facilitar criamos um sript para instalar as dependências.

In [ ]:
%pip install swig
%pip install gymnasium[box2d]
%pip install stable-baselines3[extra]

## Importar as bibliotecas 📦

In [ ]:

import gymnasium
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

## Entenda o Gymnasium e como ele funciona 🤖

🏋 A biblioteca contendo o nosso ambiente é chamada Gymnasium.
**Você utilizará bastante o Gymnasium no Deep Reinforcement Learning(Deep RL).**

Gymnasium é a **nova versão da biblioteca Gym** [mantida pela Farama Foundation](https://farama.org/).

A biblioteca do Gymnasium oferece duas coisas:

- Uma interface que permite **criar ambientes RL**.
- Uma **coleção de ambientes** (gym-control, atari, box2D...).

Vejamos um exemplo, mas primeiro vamos relembrar o loop RL.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit1/images/RL_process_game.jpg" alt="The RL process" width="100%">
<span style="font-size:80%">
Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Em cada etapa:
- Nosso agente recebe um **state (S0)** do **Environment** — recebemos o primeiro frame do nosso jogo (_environment_).
- Com base nesse **state (S0)**, o agente realiza uma **action (A0)** — nosso agente se moverá para a direita.
- O ambiente transita para um **novo** **state (S1)** — novo frame.
- O ambiente dá alguma **reward (R1)** ao agente — não estamos mortos *(Recompensa positiva +1)*.



Com Gymnasium:

1️⃣ Nós criamos um ambiente usando `gymnasium.make()`

2️⃣ Resetamos o ambiente para seu estado inicial com `observation = env.reset()`

A cada etapa:

3️⃣ Obtenha uma _action_ usando nosso modelo (em nosso exemplo, tomamos uma ação aleatória)

4️⃣ Usando `env.step(action)`, executamos essa _action_ no ambiente e recebemos
- `observation`: O novo estado (st+1)
- `reward`: A recompensa que recebemos apos a execução da ação
- `terminated`: Indica se o episódio foi encerrado (o agente atingiu o _terminal state_)
- `truncated`: Introduzido com esta nova versão, indica um limite de tempo ou se um agente sai dos limites do ambiente, por exemplo.
- `info`: Um dicionário que fornece informações adicionais (depende do ambiente).

Para mais explicações confira aqui 👉 https://gymnasium.farama.org/api/env/#gymnasium.Env.step

Se o episódio for encerrado:
- Reiniciamos o ambiente para seu estado inicial com `observation = env.reset()`

**Vamos ver um exemplo!** Certifique-se de ler o código

In [ ]:
import gymnasium as gym

# Primeiro, criamos nosso ambiente chamado LunarLander-v2
env = gym.make("LunarLander-v3")

# Em seguida, resetamos esse ambiente
observation, info = env.reset()

for _ in range(20):
  # Executa uma ação aleatória
  action = env.action_space.sample()
  print("Ação executada:", action)

  # Realiza essa ação no ambiente e obtém
  # próximo_estado, recompensa, terminado, truncado e info
  observation, reward, terminated, truncated, info = env.step(action)

  # Se o jogo terminou (pousou, colidiu) ou foi truncado (tempo esgotado)
  if terminated or truncated:
      # Reseta o ambiente
      print("Ambiente foi resetado")
      observation, info = env.reset()

env.close()


## Criar o ambiente LunarLander 🌛 E entender como ele funciona

### [O ambiente 🎮](https://gymnasium.farama.org/environments/box2d/lunar_lander/)

Neste primeiro tutorial, vamos treinar nosso agente no ambiente [Lunar Lander](https://gymnasium.farama.org/environments/box2d/lunar_lander/), **para pousar corretamente na lua**. Para isso, o agente precisa aprender **a adaptar sua velocidade e posição (horizontal, vertical e angular) para pousar corretamente.**

---


💡 Um bom hábito quando você começa a usar um ambiente é verificar sua documentação.

👉 https://gymnasium.farama.org/environments/box2d/lunar_lander/

---


Vamos dar uma olhada em como o ambiente se parece:


In [ ]:
# Criamos nosso ambiente com gym.make("<nome_do_ambiente>")
env = gym.make("LunarLander-v3")
env.reset()
print("_____OBSERVATION SPACE_____ \n")
print("Dimensões do Observation Space", env.observation_space.shape)
print("Observação amostrada aleatoriamente", env.observation_space.sample())

Vemos com `Observation Space Shape (8,)` que a observação é um vetor de tamanho 8, onde cada valor contém informações diferentes sobre o módulo de pouso:
- Coordenada horizontal da plataforma (x)
- Coordenada vertical da plataforma (y)
- Velocidade horizontal (x)
- Velocidade vertical (y)
- Ângulo
- Velocidade angular
- Se o ponto de contato da perna esquerda tocou o solo (booleano)
- Se o ponto de contato da perna direita tocou o solo (booleano)


In [ ]:
print("\n _____ACTION SPACE_____ \n")
print("Formato do Action Space", env.action_space.n)
print("Action Space amostrada aleatoriamente", env.action_space.sample()) # Executar uma ação randômica

O action space (o conjunto de ações possíveis que o agente pode realizar) é discreto, com 4 ações disponíveis. 🎮:

- Action 0: Não fazer nada,
- Action 1: Acionar o motor de orientação esquerdo,
- Action 2: Acionar o motor principal,
- Action 3: Acionar o motor de orientação direito.

Reward function (a função que dará uma recompensa a cada intervalo de tempo) 💰:

Após cada etapa, é concedida uma recompensa. A recompensa total de um episódio é a **soma das recompensas de todas as etapas desse episódio**.

Para cada etapa, a recompensa:

- É aumentada/diminuída quanto mais próximo/distante o módulo de pouso estiver da plataforma de pouso.
- É aumentada/diminuída quanto mais lento/rápido o módulo de pouso estiver se movendo.
- É diminuída quanto mais o módulo de pouso estiver inclinado (ângulo não horizontal).
- É aumentada em 10 pontos para cada perna que estiver em contato com o solo.
- É diminuída em 0,03 pontos a cada quadro em que um motor lateral estiver funcionando.
- É diminuída em 0,3 pontos a cada quadro em que o motor principal estiver funcionando.

O episódio recebe uma **recompensa adicional de -100 ou +100 pontos por colidir ou pousar com segurança, respectivamente.**

Um episódio é **considerado uma solução se obtiver pelo menos 200 pontos.**

#### Ambiente Vetorizado (_Vectorized Environment_)

- Criamos um ambiente vetorizado (um método para empilhar vários ambientes independentes em um único ambiente) de 16 ambientes, dessa forma, **teremos experiências mais diversificadas durante o treinamento.**

In [ ]:
# Cria o ambiente vetorizado com 16 ambientes
# Isso é útil para treinar o agente em múltiplos ambientes simultaneamente, aumentando a
# diversidade das experiências de treinamento e acelerando o processo de aprendizado.
env = make_vec_env('LunarLander-v3', n_envs=16)

## Criar o Modelo 🤖
- Estudamos nosso ambiente e entendemos o problema: **ser capaz de pousar o Lunar Lander na plataforma de pouso corretamente, controlando os motores de orientação esquerdo, direito e principal**. Agora vamos construir o algoritmo que vamos usar para resolver esse problema 🚀.

- Para isso, vamos usar nossa primeira biblioteca Deep RL, [Stable Baselines3 (SB3)](https://stable-baselines3.readthedocs.io/en/master/).

- SB3 é um conjunto de **implementações confiáveis de algoritmos de aprendizado por reforço em PyTorch**.

---

💡 Um bom hábito ao usar uma nova biblioteca é mergulhar primeiro na documentação: https://stable-baselines3.readthedocs.io/en/master/ e, em seguida, experimentar alguns tutoriais.

----

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit1/images/sb3.png" alt="The RL process" width="100%">
<span style="font-size:80%">
Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Para resolver esse problema, vamos usar o SB3 **PPO**. [O PPO (também conhecido como Proximal Policy Optimization) é um dos algoritmos SOTA (state of the art) de Deep Reinforcement Learning que você estudará durante este curso](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html#example%5D).

O PPO é uma combinação de:
- *Value-based reinforcement learning method*: aprender uma função de valor de ação que nos dirá a **ação mais valiosa a ser tomada, dado um estado e uma ação**.
- *Policy-based reinforcement learning method*: aprender uma *policy* que nos **dará uma distribuição de probabilidade sobre as ações**.

O Stable-Baselines3 é fácil de configurar:

1️⃣ Você **cria seu ambiente** (no nosso caso, isso foi feito acima)

2️⃣ Você define o **modelo que deseja usar e instancia esse modelo** `model = PPO(“MlpPolicy”)`

3️⃣ Você **treina o agente** com `model.learn` e define o número de etapas de treinamento

```
# Criar ambiente
env = gym.make(‘LunarLander-v2’)

# Instancie o agente
model = PPO(‘MlpPolicy’, env, verbose=1)
# Treine o agente
model.learn(total_timesteps=int(2e5))
```

In [ ]:
# Adicionamos alguns parâmetros para acelerar o treinamento
model = PPO(
    policy = 'MlpPolicy',
    env = env,
    n_steps = 1024,
    batch_size = 64,
    n_epochs = 4,
    gamma = 0.999,
    gae_lambda = 0.98,
    ent_coef = 0.01,
    verbose=1)

## Treine o agente PPO 🏃
- Vamos treinar nosso agente por 1.000.000 de etapas temporais, não se esqueça de usar a GPU no Colab. Isso levará aproximadamente 20 minutos, mas você pode usar menos etapas temporais se quiser apenas experimentar.
- Durante o treinamento, faça uma pausa para um café, você merece 🤗

In [ ]:
# Treine por 1.000.000 de etapas temporais
model.learn(total_timesteps=1000000)
# Salve o modelo
model_name = "ppo-LunarLander-v3"
model.save(model_name)

## Avalie o agente 📈
- Lembre-se de envolver o ambiente em um [Monitor](https://stable-baselines3.readthedocs.io/en/master/common/monitor.html).
- Agora que nosso agente Lunar Lander está treinado 🚀, precisamos **verificar seu desempenho**.
- O Stable-Baselines3 fornece um método para fazer isso: `evaluate_policy`.
- Para preencher essa parte, você precisa [verificar a documentação](https://stable-baselines3.readthedocs.io/en/master/guide/examples.html#basic-usage-training-saving-loading)


💡 Ao avaliar seu agente, você não deve usar seu ambiente de treinamento, mas criar um ambiente de avaliação.

In [ ]:
#@title
eval_env = Monitor(gym.make("LunarLander-v3", render_mode='rgb_array'))
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

In [ ]:
from matplotlib import animation
from IPython.display import HTML
import matplotlib.pyplot as plt

# Gera um vídeo mostrando o agente treinado atuando no ambiente de avaliação.
# O vídeo é criado a partir dos frames renderizados do ambiente enquanto o agente executa ações preditas pelo modelo.

# frames: lista de imagens (frames) capturadas do ambiente durante um episódio.
# eval_env: ambiente de avaliação já monitorado e configurado para renderizar em modo 'rgb_array'.
# model: agente treinado (PPO) que irá prever as ações a serem tomadas.

# O loop executa um episódio completo, coletando cada frame do ambiente.
# A função display_video exibe a animação dos frames capturados diretamente no notebook.

frames = []
obs = eval_env.reset()[0]
done = False

while not done:
    # O modelo prevê a próxima ação a ser tomada com base na observação atual
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    frame = eval_env.render()
    frames.append(frame)
    done = terminated or truncated

eval_env.close()

def display_video(frames):
    """
    Exibe uma animação dos frames capturados do ambiente.

    Parâmetros:
        frames (list): Lista de arrays de imagem (frames) do ambiente.

    Retorna:
        HTML: Animação em HTML para visualização no Jupyter Notebook.
    """
    fig = plt.figure(figsize=(6, 6))
    plt.axis('off')
    im = plt.imshow(frames[0])

    def animate(i):
        im.set_array(frames[i])
        return [im]

    anim = animation.FuncAnimation(fig, animate, frames=len(frames), interval=50, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

display_video(frames)

## Some additional challenges 🏆
The best way to learn **is to try things by your own**! As you saw, the current agent is not doing great. As a first suggestion, you can train for more steps. With 1,000,000 steps, we saw some great results!

In the [Leaderboard](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard) you will find your agents. Can you get to the top?

Here are some ideas to achieve so:
* Train more steps
* Try different hyperparameters for `PPO`. You can see them at https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html#parameters.
* Check the [Stable-Baselines3 documentation](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html) and try another model such as DQN.
* **Push your new trained model** on the Hub 🔥

**Compare the results of your LunarLander-v2 with your classmates** using the [leaderboard](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard) 🏆

Is moon landing too boring for you? Try to **change the environment**, why not use MountainCar-v0, CartPole-v1 or CarRacing-v0? Check how they work [using the gym documentation](https://www.gymlibrary.dev/) and have fun 🎉.

## Alguns desafios adicionais 🏆
A melhor maneira de aprender **é experimentar por conta própria**! Como você viu, o agente atual não está indo muito bem. Como primeira sugestão, você pode treinar para mais etapas. Com 1.000.000 de etapas, vimos ótimos resultados!


Aqui estão algumas ideias para conseguir isso:
* Treine mais passos
* Experimente diferentes hiperparâmetros para `PPO`. Você pode vê-los em https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html#parameters.
* Verifique a [documentação do Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html) e tente outro modelo, como o DQN.

A aterrissagem na Lua é muito chata para você? Tente **mudar o ambiente**, por que não usar MountainCar-v0, CartPole-v1 ou CarRacing-v0? Verifique como eles funcionam [usando a documentação do gym](https://www.gymlibrary.dev/) e divirta-se 🎉.